In [ ]:
from botorch.test_functions.synthetic import Branin
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from omegaconf import OmegaConf

from data.synthetic.synthetic import *
from data.synthetic.synthetic_functions import *
from gp.dsoft_ki.train import train_gp
import gp.soft_gp.train as softki_train
from gp.util import flatten_dataset, split_dataset

# Branin

In [ ]:
branin = Branin()
print(branin._bounds)
x = torch.linspace(-5, 10, 100)
y = torch.linspace(0, 15, 100)
X, Y = torch.meshgrid(x, y)
points = torch.stack([X.flatten(), Y.flatten()], dim=-1)
Z = branin(points).view(100, 100).cpu().detach().numpy()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor='none')

# Normalized Branin

In [ ]:
Z = branin(points).view(100, 100).cpu().detach().numpy()
normalize(Z, derivative=False)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor='none')

# Dataset

In [ ]:
dataset = BraninDataset(10000)
xs = np.array([x for x, y in dataset])
ys = np.array([y['energy'].item() for x , y in dataset])

x1 = np.array([x[0] for x in xs])
x2 = np.array([x[1] for x in xs])

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x1, x2, ys, c=ys, cmap='viridis')

# Normalized Branin with Rescaled Dataset

In [ ]:
dataset = BraninDataset(10000)
xs = np.array([x for x, y in dataset])
ys = np.array([y['energy'].item() for x , y in dataset])

lb = np.array([item[0] for item in branin._bounds])
ub = np.array([item[1] for item in branin._bounds])
scaled_xs = from_unit_cube(xs, lb, ub)
x1 = np.array([x[0] for x in scaled_xs])
x2 = np.array([x[1] for x in scaled_xs])

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

Z = branin(points).view(100, 100).cpu().detach().numpy()
normalize(Z, derivative=False)
ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor='none', alpha=0.5)
ax.scatter(x1, x2, ys, c=ys, cmap='viridis')

In [ ]:
train_dataset, val_dataset, test_dataset = split_dataset(
    dataset,
    train_frac=0.9,
    val_frac=0.0
)

CONFIG = OmegaConf.create({
    'model': {
        'name': 'soft-gp',
        'kernel': {
            '_target_': 'RBFKernel'
        },
        'use_scale': False,
        'num_inducing': 512,
        'induce_init': 'kmeans',
        'noise': 1e-3,
        'deriv_noise': 1e-3,
        'learn_noise': False,
        'solver': 'solve',
        'cg_tolerance': 1e-5,
        'mll_approx': 'hutchinson',
        'fit_chunk_size': 1024,
        'use_qr': False,
        'dtype': 'float32',
        'device': 'cuda:0',
    },
    'dataset': {
        'name': 'Sine',
        'num_workers': 1,
        'train_frac': 0.9,
        'val_frac': 0.0,
    },
    'synthetic': {
        'N': 3,
    },
    'training': {
        'seed': 42,
        'batch_size': 256,
        'learning_rate': 1e-2,
        'epochs': 50,
    },
    'wandb': {
        'watch': False,
        'group': 'test',
        'entity': 'bogp',
        'project': 'dsoft-ki',
    }
})

"""
w(x1, z1)  w(x1, z2)  w(x1, z3)
w(x2, z1)  w(x2, z2)  w(x2, z3)
dw/1(x1, z1)  dw/1(x1, z2)  dw/1(x1, z3)
dw/2(x1, z1)  dw/2(x1, z2)  dw/2(x1, z3)
dw/1(x2, z1)  dw/1(x2, z2)  dw/1(x2, z3)
dw/2(x2, z1)  dw/2(x2, z2)  dw/2(x2, z3)
"""

"""
torch.Size([4, 3, 2])
B x M x D -> B x D x M
"""


dsoftki_gp = train_gp(CONFIG, train_dataset, test_dataset)


In [ ]:
device = "cuda:0"
xs = torch.stack([x for x, y in dataset])
pred_ys = dsoftki_gp.pred(xs.to(device))[:len(xs)].detach().cpu().numpy()

lb = np.array([item[0] for item in branin._bounds])
ub = np.array([item[1] for item in branin._bounds])
scaled_xs = from_unit_cube(xs, lb, ub)
x1 = np.array([x[0] for x in scaled_xs])
x2 = np.array([x[1] for x in scaled_xs])

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

Z = branin(points).view(100, 100).cpu().detach().numpy()
normalize(Z, derivative=False)
ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor='none', alpha=0.5)
ax.scatter(x1, x2, pred_ys, c=pred_ys, cmap='viridis', s=0.1)

# Soft GP

In [ ]:
dataset = BraninDataset(10000, with_deriv=False)
xs = np.array([x for x, y in dataset])
ys = np.array([y for x , y in dataset])

lb = np.array([item[0] for item in branin._bounds])
ub = np.array([item[1] for item in branin._bounds])
scaled_xs = from_unit_cube(xs, lb, ub)
x1 = np.array([x[0] for x in scaled_xs])
x2 = np.array([x[1] for x in scaled_xs])

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

Z = branin(points).view(100, 100).cpu().detach().numpy()
normalize(Z, derivative=False)
ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor='none', alpha=0.5)
ax.scatter(x1, x2, ys, c=ys, cmap='viridis', s=0.1)

# SoftKI

In [ ]:
train_dataset, val_dataset, test_dataset = split_dataset(
    dataset,
    train_frac=0.9,
    val_frac=0.0
)

CONFIG = OmegaConf.create({
    'model': {
        'name': 'soft-gp',
        'kernel': {
            '_target_': 'RBFKernel'
        },
        'T': 0.005,
        'use_T': False,
        'learn_T': False,
        'use_scale': False,
        'threshold': 0.005,
        'learn_threshold': False,
        'use_threshold': False,
        'num_inducing': 512,
        'induce_init': 'kmeans',
        'noise': 1e-3,
        'learn_noise': False,
        'solver': 'solve',
        'cg_tolerance': 1e-5,
        'mll_approx': 'hutchinson',
        'fit_chunk_size': 64,
        'use_qr': False,
        'dtype': 'float32',
        'device': 'cuda:0',
    },
    'dataset': {
        'name': 'Sine',
        'num_workers': 1,
        'train_frac': 0.9,
        'val_frac': 0.0,
    },
    'synthetic': {
        'N': 10000,
    },
    'training': {
        'seed': 42,
        'batch_size': 256,
        'learning_rate': 0.01,
        'epochs': 50,
    },
    'wandb': {
        'watch': False,
        'group': 'test',
        'entity': 'bogp',
        'project': 'dsoft-ki',
    }
})

softki_gp = softki_train.train_gp(CONFIG, train_dataset, test_dataset)

In [ ]:
xs = torch.stack([x for x, y in dataset])
pred_ys = softki_gp.pred(xs.to(device))[:len(xs)].detach().cpu().numpy()

lb = np.array([item[0] for item in branin._bounds])
ub = np.array([item[1] for item in branin._bounds])
scaled_xs = from_unit_cube(xs, lb, ub)
x1 = np.array([x[0] for x in scaled_xs])
x2 = np.array([x[1] for x in scaled_xs])

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

Z = branin(points).view(100, 100).cpu().detach().numpy()
normalize(Z, derivative=False)
ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor='none', alpha=0.5)
ax.scatter(x1, x2, pred_ys, c=ys, cmap='viridis', s=0.1)